# TRABALHO 3: REGRESSÃO LOGÍSTICA E SVM
**Integrantes:** Julia Welter, Pedro Pacheco, Pedro Albuquerque, Tiago Melo

In [1]:
import pickle
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, precision_score, recall_score

import utils

# PARTE 3: Regressão Logística

### Questões 1-6: Base 'risco_credito.pkl'

In [2]:
# 1. Carregar base
df_pequeno = utils.RiscoCredito.carregar('risco_credito.pkl')

# 2. Encoder e remover 'moderado'
df_pequeno = utils.RiscoCredito.remover_moderado(df_pequeno)
X_p = utils.RiscoCredito.encode_previsores(df_pequeno)
y_p, le_p = utils.RiscoCredito.encode_classe(df_pequeno)

# 3. Treinar Regressão Logística (random_state=1)
modelo_lr_p = LogisticRegression(random_state=1)
modelo_lr_p.fit(X_p, y_p)

# 4. Intercepto (B0)
print(f"4. Intercepto (B0): {modelo_lr_p.intercept_[0]:.8f}")

# 5. Coeficientes
print(f"5. Coeficientes: {modelo_lr_p.coef_}")

# 6. Predict
cliente_a = utils.RiscoCredito.novo_cliente(2, 1, 0, 2) # boa, alta, nenhuma, >35
pred_a = le_p.inverse_transform(modelo_lr_p.predict(cliente_a))
print(f"6a. Cliente A -> Risco previsto: {pred_a[0]}")

cliente_b = utils.RiscoCredito.novo_cliente(0, 1, 1, 0) # ruim, alta, adequada, <15
pred_b = le_p.inverse_transform(modelo_lr_p.predict(cliente_b))
print(f"6b. Cliente B -> Risco previsto: {pred_b[0]}")

4. Intercepto (B0): -0.80855152
5. Coeficientes: [[-0.76706797  0.23921064 -0.47989768  1.12196145]]
6a. Cliente A -> Risco previsto: baixo
6b. Cliente B -> Risco previsto: alto


### Questões 7-8: Base 'credit.pkl'

In [3]:
with open('credit.pkl', 'rb') as f:
    X_train, y_train, X_test, y_test = pickle.load(f)

# 7. Aplicar Regressão Logística
modelo_lr = LogisticRegression(random_state=1)
modelo_lr.fit(X_train, y_train)
y_pred_lr = modelo_lr.predict(X_test)
print(f"7. Taxa de acerto (Acurácia): {accuracy_score(y_test, y_pred_lr)*100:.2f}%")

# 8. Análise de Resultados
print("\n8. Relatório de Classificação:")
print(classification_report(y_test, y_pred_lr))

7. Taxa de acerto (Acurácia): 94.60%

8. Relatório de Classificação:
              precision    recall  f1-score   support

           0       0.97      0.97      0.97       436
           1       0.79      0.78      0.79        64

    accuracy                           0.95       500
   macro avg       0.88      0.88      0.88       500
weighted avg       0.95      0.95      0.95       500



# PARTE 4: SVM

### 1-3. SVM com Kernel Linear

In [4]:
# 1. Aplicar algoritmo SVM
# 2. Kernel linear, C=1.0, random_state=1
svm_linear = SVC(kernel='linear', C=1.0, random_state=1)
svm_linear.fit(X_train, y_train)

# 3. Acurácia (Esperado: 0.946)
y_pred_linear = svm_linear.predict(X_test)
print(f"3. Acurácia (Kernel Linear): {accuracy_score(y_test, y_pred_linear):.3f}")

3. Acurácia (Kernel Linear): 0.946


### 4. Teste de demais Kernels

In [5]:
kernels = ['poly', 'sigmoid', 'rbf']
for k in kernels:
    svm = SVC(kernel=k, C=1.0, random_state=1)
    svm.fit(X_train, y_train)
    acc = accuracy_score(y_test, svm.predict(X_test))
    print(f"Kernel {k:7} -> Acurácia: {acc:.3f}")

print("\nO melhor kernel para esta base de dados é o RBF.")

Kernel poly    -> Acurácia: 0.968
Kernel sigmoid -> Acurácia: 0.838
Kernel rbf     -> Acurácia: 0.982

O melhor kernel para esta base de dados é o RBF.
